<a href="https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will start with a Decision Tree classifier because it is relatively easy to interpret and fits my content refresh prioritization lane. The goal is to rank content items for review, not to prove that refreshing them will improve performance. I will compare its ranking with my Week 4 rule-based baseline using Precision@20 and Precision@50 on the same held-out test set. I will exclude trend and decline-derived fields from the model features to avoid leakage.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

data_url = (
    "https://raw.githubusercontent.com/"
    "sahanabytes/machine-learning-portfolio/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

print("Dataset shape:", df.shape)
print("\nRequired columns check:")

required = [
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
]

for col in required:
    print(f"{col}: {col in df.columns}")

print("\nTrend labels:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nNumber of clients:", df["client_id"].nunique())


Dataset shape: (30000, 44)

Required columns check:
client_id: True
days_since_last_update: True
impressions_90d: True
trend_direction: True

Trend labels:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Number of clients: 32


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will split the data by `client_id` so that each client's rows appear in only one split. This reduces the risk of testing on clients the model has already seen. I will use the same held-out test rows to evaluate both the Decision Tree and the Week 4 baseline. I will report Precision@20 and Precision@50 so the comparison matches the baseline's ranking task.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

# Define the observed outcome for evaluation.
df["decline_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Hold out entire clients so they cannot appear in both sets.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, y=df["decline_label"], groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())
print(
    "Clients overlap:",
    set(train_df["client_id"]) & set(test_df["client_id"])
)
print("\nDecline rate — train:", round(train_df["decline_label"].mean(), 3))
print("Decline rate — test:", round(test_df["decline_label"].mean(), 3))

assert set(train_df["client_id"]).isdisjoint(set(test_df["client_id"]))



Training rows: 22885
Testing rows: 7115
Training clients: 24
Testing clients: 8
Clients overlap: set()

Decline rate — train: 0.55
Decline rate — test: 0.517


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [21]:

from sklearn.tree import DecisionTreeClassifier

# Define the target: observed decline, not refresh success.
feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "search_volume",
    "competition",
    "word_count",
    "content_age_days",
    "avg_position",
    "ctr",
]

X_train = train_df[feature_cols].apply(pd.to_numeric, errors="coerce")
X_test = test_df[feature_cols].apply(pd.to_numeric, errors="coerce")

# Fit missing-value replacements on training data only.
train_medians = X_train.median()
X_train = X_train.fillna(train_medians).fillna(0)
X_test = X_test.fillna(train_medians).fillna(0)

y_train = train_df["decline_label"]
y_test = test_df["decline_label"]

model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=100,
    random_state=42,
)
model.fit(X_train, y_train)

# Apply the Week 4 baseline to the exact same test rows.
baseline_score = (
    (test_df["days_since_last_update"] >= 180).astype(int) * 50
    + (test_df["impressions_90d"] >= 500).astype(int) * 50
)

model_score = model.predict_proba(X_test)[:, 1]

def precision_at_k(y_true, scores, k):
    top_indices = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[top_indices].mean()

results = pd.DataFrame([
    {
        "method": "Week 4 baseline",
        "Precision@20": precision_at_k(y_test, baseline_score, 20),
        "Precision@50": precision_at_k(y_test, baseline_score, 50),
    },
    {
        "method": "Decision Tree",
        "Precision@20": precision_at_k(y_test, model_score, 20),
        "Precision@50": precision_at_k(y_test, model_score, 50),
    },
])

print("Model vs baseline — held-out clients")
print(results.round(3).to_string(index=False))

print("\nModel feature importance:")
print(
    pd.Series(model.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
    .round(3)
    .to_string()
)


Model vs baseline — held-out clients
         method  Precision@20  Precision@50
Week 4 baseline          0.60          0.60
  Decision Tree          0.45          0.56

Model feature importance:
impressions_90d           0.554
content_age_days          0.248
avg_position              0.120
ctr                       0.078
word_count                0.000
competition               0.000
days_since_last_update    0.000
search_volume             0.000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [22]:

error_review = test_df[
    ["content_id", "decline_label"]
].copy()

error_review["predicted_decline"] = model.predict(X_test)
error_review["predicted_probability"] = model.predict_proba(X_test)[:, 1]

error_review["result"] = np.select(
    [
        (error_review["decline_label"] == 1)
        & (error_review["predicted_decline"] == 1),
        (error_review["decline_label"] == 0)
        & (error_review["predicted_decline"] == 0),
        (error_review["decline_label"] == 0)
        & (error_review["predicted_decline"] == 1),
    ],
    ["true_positive", "true_negative", "false_positive"],
    default="false_negative",
)

print("Prediction error summary:")
print(error_review["result"].value_counts().to_string())

for error_type in ["false_positive", "false_negative"]:
    print(f"\nExample {error_type.replace('_', ' ')} rows:")
    print(
        error_review[error_review["result"] == error_type]
        .sort_values("predicted_probability", ascending=(error_type == "false_negative"))
        .head(5)[
            ["content_id", "decline_label", "predicted_probability", "result"]
        ].to_string(index=False)
    )


Prediction error summary:
result
true_positive     3242
false_positive    2664
true_negative      776
false_negative     433

Example false positive rows:
          content_id  decline_label  predicted_probability         result
content_be5e23c0a35e              0               0.720091 false_positive
content_e3de4ddde194              0               0.720091 false_positive
content_e906960b0d12              0               0.720091 false_positive
content_f452475bff46              0               0.720091 false_positive
content_be106cd29636              0               0.720091 false_positive

Example false negative rows:
          content_id  decline_label  predicted_probability         result
content_7bc32bc1df59              1               0.013569 false_negative
content_4595e8704e07              1               0.187335 false_negative
content_8c482a64a3df              1               0.187335 false_negative
content_f68fc903dd4f              1               0.187335 false_negative
c

### Errors and interpretation

The Week 4 baseline performed better than the Decision Tree on the held-out clients. Precision@20 was 0.60 for the baseline and 0.45 for the tree; Precision@50 was 0.60 and 0.56, respectively.

The tree relied most on `impressions_90d` (importance 0.554), followed by `content_age_days` (0.248), `avg_position` (0.120), and `ctr` (0.078). The remaining listed features had zero importance in this fitted tree.

The error analysis found 2,664 false positives and 433 false negatives. The tree flags some pages as likely to decline when they are labelled otherwise, and it misses some observed declines. The Week 4 baseline remains the stronger ranking rule in this experiment. These results are directional because the test set contains eight held-out clients. Predicting observed decline does not prove that refreshing a page will improve its performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.